# Avant tout…

Avant de lancer ce fichier, assurez-vous d’avoir placé les fichiers [train.csv](../data/train.csv), [test.csv](../data/test.csv) et [sample_submission.csv](../data/sample_submission.csv) dans le dossier "data".

In [1]:
import pandas as pd
import numpy as np

In [2]:
train_dataset = pd.read_csv('../data/train.csv')
to_predict = pd.read_csv('../data/test.csv')

y_index = train_dataset.columns.get_loc("SalePrice")
print(y_index + 1 == train_dataset.columns.size) # Si vrai, SalePrice est la dernière colonne

True


In [3]:
X_train_dataset = train_dataset.iloc[:, :-1].values
y_train_dataset = train_dataset.iloc[:, -1].values
X_to_predict_dataset = to_predict.iloc[:, :].values # Pas besoin d’enlever la dernière colonne, elle n’existe pas (on doit la prédire)

In [4]:
# Find categorical variables

train_dataset_df = pd.DataFrame(X_train_dataset)
train_dataset_df.columns = train_dataset.columns[:-1]

categorical_cols = train_dataset_df.select_dtypes(include=['str']).columns
print(categorical_cols.size)

X_train_dataset = train_dataset_df.iloc[:, :].values

43


In [28]:
num_col = train_dataset_df.columns.difference(categorical_cols)

num_col.size

37

In [5]:
# Handle categorical variable

train_dataset_df = pd.get_dummies(train_dataset_df, columns=categorical_cols)
X_train_dataset = train_dataset_df.iloc[:, :-1].values

In [6]:
print(X_train_dataset[1])

[2 20 80.0 9600 6 8 1976 1976 0.0 978 0 284 1262 1262 0 0 1262 0 1 2 0 3 1
 6 1 1976.0 2 460 298 0 0 0 0 0 0 5 2007 False False False True False
 False True False False False False False True False False False True True
 False False False True False False True False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False True False True
 False False False False False False False False False True False False
 False False False True False False False False False False True False
 False False False False False True False False False False False True
 False False False False False False False False False False False False
 False False True False False False False False False False False False
 False False False False False True False False False False False False
 False False False False False False False True False False False False
 True False True False False False False False False True Fals

In [7]:
# Handle missing values
from sklearn.impute import SimpleImputer

imp_mean = SimpleImputer(missing_values = np.nan, strategy = 'mean')
X_train_dataset = imp_mean.fit_transform(X_train_dataset)

print(np.isnan(X_train_dataset).any())

False


In [8]:
# Divide : Train and test set

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_train_dataset, y_train_dataset, test_size = 0.2, random_state = 0)

print (X_train, X_test, y_train, y_test)

[[6.19000000e+02 2.00000000e+01 9.00000000e+01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [8.71000000e+02 2.00000000e+01 6.00000000e+01 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]
 [9.30000000e+01 3.00000000e+01 8.00000000e+01 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]
 ...
 [1.21700000e+03 9.00000000e+01 6.80000000e+01 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]
 [5.60000000e+02 1.20000000e+02 7.00499584e+01 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]
 [6.85000000e+02 6.00000000e+01 5.80000000e+01 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]] [[5.30000000e+02 2.00000000e+01 7.00499584e+01 ... 1.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [4.92000000e+02 5.00000000e+01 7.90000000e+01 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]
 [4.60000000e+02 5.00000000e+01 7.00499584e+01 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]
 ...
 [1.38800000e+03 5.00000000e+01 6.00000000e+01 ... 0.00000000e+00
  1.00000000e+00 0.00000000e+00]

In [9]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

X_train

array([[-0.28399978, -0.86836547,  0.9786782 , ..., -0.07764847,
        -0.10188534, -2.20665963],
       [ 0.31396549, -0.86836547, -0.46715589, ..., -0.07764847,
        -0.10188534,  0.45317365],
       [-1.53213363, -0.63114155,  0.4967335 , ..., -0.07764847,
        -0.10188534,  0.45317365],
       ...,
       [ 1.13498129,  0.79220197, -0.08160014, ..., -0.07764847,
        -0.10188534,  0.45317365],
       [-0.42399958,  1.50387373,  0.01719652, ..., -0.07764847,
        -0.10188534,  0.45317365],
       [-0.12738983,  0.08053021, -0.56354483, ..., -0.07764847,
        -0.10188534,  0.45317365]], shape=(1168, 287))

In [10]:
y_train = np.load("../data/processed_data/y_train.npy")
X_train = np.load("../data/processed_data/X_train.npy")
i = np.where(y_train == 208500) #prix de la premiere ligne à l’origine (notamment)
print(X_train[i, 0]) # id de la première ligne (1)

i = np.where(y_train == 140000) #prix de la 4e ligne à l’origine (notamment)
print(X_train[i, 0]) # id de la 4e ligne (4)


[[1.]]
[[ 248.  704.  970.  675.  896.  201.  841. 1412.  594.  355.    4.   64.
   503.  551. 1298. 1119.  702. 1209.]]


In [3]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

preprocess = joblib.load("preprocess.joblib")

dataset = pd.read_csv('../data/train.csv')
to_predict = pd.read_csv('../data/test.csv')

In [34]:
X_dataset = dataset.iloc[:, :-1]
y_dataset = pd.DataFrame(dataset.iloc[:, -1])

In [22]:
X_dataset = preprocess.transform(dataset)
X_to_predict = preprocess.transform(to_predict)

/home/emmy/dev/projects/kaggle/house-pricing-beginner/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/emmy/dev/projects/kaggle/house-pricing-beginner/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [37]:
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, test_size = 0.2, random_state = 0)
X_train

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
618,619,20,RL,90.0,11694,Pave,NaN,Reg,Lvl,AllPub,...,260,0,NaN,NaN,NaN,0,7,2007,New,Partial
870,871,20,RL,60.0,6600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,8,2009,WD,Normal
92,93,30,RL,80.0,13360,Pave,Grvl,IR1,HLS,AllPub,...,0,0,NaN,NaN,NaN,0,8,2009,WD,Normal
817,818,20,RL,NaN,13265,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,7,2008,WD,Normal
302,303,20,RL,118.0,13704,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,1,2006,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
763,764,60,RL,82.0,9430,Pave,NaN,Reg,Lvl,AllPub,...,180,0,NaN,NaN,NaN,0,7,2009,WD,Normal
835,836,20,RL,60.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2010,WD,Normal
1216,1217,90,RM,68.0,8930,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
559,560,120,RL,NaN,3196,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,10,2006,WD,Normal
